In [ ]:
import json
from sentence_transformers import SentenceTransformer, util
import torch
import statistics
from tqdm import tqdm
import pandas as pd

In [ ]:
with open('wiki_test_corpus_full.json', 'r') as file:
    data = json.load(file)

In [ ]:
model = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-cased-v2', 
                            device = 0) # optional define GPU

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu') # optional, define GPU
fin_res = []
for el in tqdm(data):
    # s_text - source text
    # t_text - target text
    # Calculate similarity of full texts
    tensor1 = model.encode(' '.join(el.get('s_text')), convert_to_tensor=True)
    tensor2 = model.encode(' '.join(el.get('t_text')), convert_to_tensor=True)
    
    tensor1 = tensor1.to(device)
    tensor2 = tensor2.to(device)
    
    sim = util.pytorch_cos_sim(tensor1, tensor2)
    sim_full = sim.item()
    
    # Calculate similarity of sentences
    res_sim_alligned = []
    res_sim_not_alligned = []
    # Encode all sentences in both lists (convert to tensors and move to correct device)
    embeddings_a = [model.encode(x, convert_to_tensor=True).to(device) for x in el.get('s_text')]
    embeddings_b = [model.encode(x, convert_to_tensor=True).to(device) for x in el.get('t_text')]

    # Compare every sentence in list_a with every sentence in list_b
    # Comparing similar and not similar sentences
    for i, emb_a in enumerate(embeddings_a):
        for j, emb_b in enumerate(embeddings_b):
            sim = util.pytorch_cos_sim(emb_a, emb_b).item()
            if i == j:
                res_sim_alligned.append(sim)
            else: 
                res_sim_not_alligned.append(sim)
    
    dct = {'s_id' : el.get('s_id'),
           't_id' : el.get('t_id'), 
           'full_sim' : sim_full, 
    'sim_not_alligned' : statistics.mean(res_sim_not_alligned),
    'sim_alligned' : statistics.mean(res_sim_alligned)
    }
    fin_res.append(dct)
fin_df = pd.DataFrame(fin_res)

In [ ]:
print(f'Mean similarity of similar sentences: {fin_df.sim_alligned.mean()}')
print(f'Mean similarity of similar sentences: {fin_df.full_sim.mean()}')
print(f'Mean similarity of full texts: {fin_df.sim_not_alligned.mean()}')